In [13]:
import qiskit_metal as qm
from qiskit_metal import designs, draw, Dict
from qiskit_metal.qlibrary.core import QComponent


In [32]:
design = designs.design_planar.DesignPlanar()
design.overwrite_enabled = True 
design._chips.main.size.size_x = '5mm'
design._chips.main.size.size_y = '5mm' 
gui = qm.MetalGUI(design)

01:28PM 49s INFO [_start_renderers]: Renderer=hfss skipped: runtime dependency not installed (QAnsysRenderer requires pyEPR. Install with: pip install 'quantum-metal[ansys]').


01:28PM 49s INFO [_start_renderers]: Renderer=q3d skipped: runtime dependency not installed (QAnsysRenderer requires pyEPR. Install with: pip install 'quantum-metal[ansys]').
01:28PM 49s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).
01:28PM 49s INFO [_start_renderers]: Renderer=aedt_q3d skipped: runtime dependency not installed (QPyaedt requires pyaedt (ansys.aedt.core). Install with: pip install 'quantum-metal[ansys]').
01:28PM 49s INFO [_start_renderers]: Renderer=aedt_hfss skipped: runtime dependency not installed (QPyaedt requires pyaedt (ansys.aedt.core). Install with: pip install 'quantum-metal[ansys]').


In [15]:
class myTransmon(QComponent):
    """Transmon qubit component.
 
    Inherits 'QComponent' class.
 
   
    Default Options:
        * trace_width: 'cpw_width' -- Width of the transmission line attached to the transmon
        * trace_gap: 'cpw_gap' -- Gap of the transmission line
        * lead_length: '25um' -- Length of the transmission line attached to the transmon
        * pad_width: '80um' -- Width of the transmon pad
        * pad_height: '80um' -- Height of the transmon pad
        * pad_gap: '58um' -- Gap of the transmon pad
        * taper_height: '122um' -- Height of the taper from the transmon to the transmission line
    """
 
    default_options = Dict(
        trace_width="10um",
        lead_length="25um",
        pad_width="80um",
        pad_height="80um",
        pad_gap="58um",
        taper_height="122um",
        orientation="90"
    )
 
 
 
 
 
    def make(self):
        """This is executed by the user to generate the qgeometry for the
        component."""
 
        p = self.p
 
        pad_width = p.pad_width
        pad_height = p.pad_height
        pad_gap = p.pad_gap
        trace_width = p.trace_width
        trace_width_half = trace_width / 2.0
        pad_width_half = pad_width / 2.0
        lead_length = p.lead_length
        taper_height = p.taper_height
        pad_gap = p.pad_gap
        #########################################################
 
        # Geometry of main transmon structure
        # The shape is a polygon and we prepare this point as orientation is 90 degree
        transmon = draw.Polygon(
            [
                (0, trace_width_half),
                (-taper_height, pad_width_half),
                (-(pad_height + taper_height), pad_width_half),
                (-(pad_height + taper_height), -pad_width_half),
                (-taper_height, -pad_width_half),
                (0, -trace_width_half),
                (lead_length, -trace_width_half),
                (lead_length, trace_width_half),
                (0, trace_width_half),
            ]
        )
 
        # Geometry pocket (gap)
        # Same way applied for pocket
        pocket = draw.Polygon(
            [
                (0, trace_width_half + pad_gap),
                (-taper_height, pad_width_half + pad_gap),
                (-(pad_height + taper_height + pad_gap), pad_width_half + pad_gap),
                (-(pad_height + taper_height + pad_gap), -(pad_width_half + pad_gap)),
                (-taper_height, -(pad_width_half + pad_gap)),
                (0, -(trace_width_half + pad_gap)),
                (lead_length, -(trace_width_half + pad_gap)),
                (lead_length, trace_width_half + pad_gap),
                (0, trace_width_half + pad_gap),
            ]
        )
 
        # These variables are used to graphically locate the pin locations
        main_pin_line = draw.LineString(
            [(lead_length, trace_width_half), (lead_length, -trace_width_half)]
        )
       
        # Define pin lines along the five main sides
        pin_left = draw.LineString([
            (-(pad_height + taper_height), pad_width_half),
            (-(pad_height + taper_height), -pad_width_half)
        ])
 
        pin_top = draw.LineString([
            (-(pad_height + taper_height), pad_width_half),
            (-taper_height, pad_width_half)
        ])
 
        pin_bottom = draw.LineString([
            (-(pad_height + taper_height), -pad_width_half),
            (-taper_height, -pad_width_half)
        ])
 
        pin_taper_top = draw.LineString([
            (-taper_height, pad_width_half),
            (0, trace_width_half)
        ])
 
        pin_taper_bottom = draw.LineString([
            (-taper_height, -pad_width_half),
            (0, -trace_width_half)
        ])
 
        # Create polygon object list
        polys1 = [main_pin_line, transmon, pocket, pin_left, pin_top, pin_bottom, pin_taper_top, pin_taper_bottom]
 
        # Rotates and translates all the objects as requested. Uses package functions in
        # 'draw_utility' for easy rotation/translation
        polys1 = draw.rotate(polys1, p.orientation, origin=(0, 0))
        polys1 = draw.translate(polys1, xoff=p.pos_x, yoff=p.pos_y)
        [main_pin_line, transmon, pocket, pin_left, pin_top, pin_bottom, pin_taper_top, pin_taper_bottom] = polys1
 
        # Adds the object to the qgeometry table
        self.add_qgeometry("poly", dict(transmon=transmon), layer=p.layer)
        self.add_qgeometry("poly", dict(pocket=pocket), layer=p.layer, subtract=True)
 
        self.add_pin(
            "readout",
            points=list(main_pin_line.coords),
            width=trace_width,
            input_as_norm=False
        )
 
       
        self.add_pin(
            "left",
            points=list(pin_left.coords),
            width=trace_width,
            input_as_norm=False
        )
 
        self.add_pin(
            "top",
            points=list(pin_top.coords),
            width=trace_width,
            input_as_norm=False
        )
 
        self.add_pin(
            "bottom",
            points=list(pin_bottom.coords),
            width=trace_width,
            input_as_norm=False
        )
 
        self.add_pin(
            "taper_top",
            points=list(pin_taper_top.coords),
            width=trace_width,
            input_as_norm=False
        )
 
        self.add_pin(
            "taper_bottom",
            points=list(pin_taper_bottom.coords),
            width=trace_width,
            input_as_norm=False
        )

In [34]:
import numpy as np
from qiskit_metal import draw, Dict
from qiskit_metal.qlibrary.core import QComponent
from airbridge_route import _fillet_centerline

ARC_POINTS = 33   # points per rounded corner (airbridge_route default is 17)


# ----------------------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------------------
def _arclength(coords):
    """Cumulative arc length along a polyline."""
    seg = np.linalg.norm(np.diff(coords, axis=0), axis=1)
    return np.concatenate([[0.0], np.cumsum(seg)])


def _s_at_point(coords, pt):
    """Arc-length position (along the polyline) of the point on it closest to `pt`."""
    pt = np.asarray(pt, dtype=float)
    cum = _arclength(coords)
    best_d, best_s = np.inf, 0.0
    for k in range(len(coords) - 1):
        a, ab = coords[k], coords[k + 1] - coords[k]
        L2 = ab @ ab
        t = 0.0 if L2 == 0 else float(np.clip((pt - a) @ ab / L2, 0.0, 1.0))
        d = np.linalg.norm(a + t * ab - pt)
        if d < best_d:
            best_d, best_s = d, cum[k] + t * np.sqrt(L2)
    return best_s


def _check_fillets(points, coords, r, tol=1e-9):
    """Guard the rounding done by _fillet_centerline.

    It leaves a corner sharp, without a word, when its arc does not fit, and it does
    not notice two neighbouring arcs overlapping. All corners here are 90 degrees, so
    every segment between two corners must be >= 2r (>= r next to a path end)."""
    pts = np.asarray(points, dtype=float)
    d = np.diff(pts, axis=0)
    seg = np.linalg.norm(d, axis=1)
    u = d / seg[:, None]
    if np.any(np.abs(np.einsum("ij,ij->i", u[:-1], u[1:])) > 1e-9):
        raise ValueError("All waveguide corners must be 90 degrees.")
    need = np.full(len(seg), 2 * r)
    need[0] -= r
    need[-1] -= r
    short = np.where(seg < need - tol)[0]
    if len(short):
        i = int(short[0])
        raise ValueError(f"Segment {tuple(pts[i])} -> {tuple(pts[i + 1])} is shorter than "
                         "the fillets at its ends need. Reduce `fillet` or lengthen "
                         "dip_width / pitch / corridor_drop / block_b_drop.")
    for v in pts[1:-1]:
        if np.any(np.all(np.abs(coords - v) < tol, axis=1)):
            raise ValueError(f"Corner at {tuple(v)} was left sharp by the fillet.")


def _rows(y_first, n, pitch, x_left, x_right, x_final):
    """Vertices for n horizontal rows stacked downward from y_first.

    Row 0 heads right from the vertex the caller has already added; rows alternate.
    n must be even so the last row heads left, ending at x_final."""
    v = []
    for k in range(n):
        y = y_first - k * pitch
        x = x_right if k % 2 == 0 else (x_left if k < n - 1 else x_final)
        v.append((x, y))
        if k < n - 1:
            v.append((x, y - pitch))
    return v


def _layout(g, nA, WA, nB, WB):
    """Full waypoint list plus the five coupling-point centres.

    Right half (chip right edge -> bottom coupling point):
      in along the top -> down onto the right shoulder (U dip) -> back up -> block A rows
      -> down the corridor -> U dip into the right edge -> down -> block B rows
      -> back under the qubit -> up into the bottom edge (U dip, shared with the left half).
    The left half is the mirror image about x = qubit_x, traversed in reverse."""
    pitch = g["pitch"]
    y_lastA = g["yU"] + g["corridor_drop"]
    y1 = y_lastA + (nA - 1) * pitch                  # first row of block A
    Y = y1 + pitch                                   # incoming run, one pitch above it
    yB0 = g["yL"] - g["block_b_drop"]                # first row of block B

    R = [(g["start_x"], Y), (g["x_a1"], Y), (g["x_a1"], g["y_dip1"]),
         (g["x_b1"], g["y_dip1"]), (g["x_b1"], y1)]
    R += _rows(y1, nA, pitch, g["x_min"], g["x_min"] + WA, g["x_cor"])
    R += [(g["x_cor"], g["yU"]), (g["x_tip2"], g["yU"]), (g["x_tip2"], g["yL"]),
          (g["x_cor"], g["yL"]), (g["x_cor"], yB0)]
    R += _rows(yB0, nB, pitch, g["x_min"], g["x_min"] + WB, g["x_p3"])
    R += [(g["x_p3"], g["y_tip3"])]

    qx = g["qx"]
    M = [(2 * qx - x, y) for x, y in reversed(R)]
    centres = [(g["x_c1"], g["y_dip1"]), (g["x_tip2"], g["yc2"]), (qx, g["y_tip3"]),
               (2 * qx - g["x_tip2"], g["yc2"]), (2 * qx - g["x_c1"], g["y_dip1"])]
    return R + M, centres


class GiantAtomWaveguide(QComponent):
    """Giant-atom waveguide coupled to the transmon at five points.

    Contact order (global frame, qubit lead pointing up), one U-shaped dip per edge,
    each dipping `contact_depth` into the pad so the trace touches the qubit:

        1 right shoulder -> 2 right edge -> 3 bottom edge -> 4 left edge -> 5 left shoulder

    Between consecutive contacts the waveguide runs `target_length` (contact centre to
    contact centre), taken up by a serpentine of horizontal rows. The waveguide enters
    at the chip's right edge above the qubit and leaves at the left edge; the layout is
    mirror-symmetric about x = qubit_x, so the qubit must sit on the chip's centre line.

    All x offsets below are measured from qubit_x, all y offsets from the pad's top edge.
    """

    default_options = Dict(
        # --- qubit geometry (match myTransmon options; orientation 90 => lead points up)
        qubit_x="0mm",            # qubit centre x (also the mirror axis)
        qubit_y="0mm",            # y of the pad's top edge (shoulder line)
        qubit_pad_width="200um",
        qubit_pad_height="175um",
        qubit_lead_width="25um",  # qubit trace_width (sets where the shoulders start)
        # --- contacts (U dips)
        dip_width="60um",         # leg-to-leg spacing of every dip (must be >= 2*fillet)
        contact_depth="0um",     # how far each dip reaches into the pad (galvanic contact)
        contact_x="62um",         # contact 1: dip centre on the right shoulder
        side_contact_y="130um",   # contacts 2 and 4: dip centre this far below the pad top
        # --- routing between contacts
        corridor_x="180um",       # vertical lane beside the pad that carries the waveguide
        corridor_drop="120um",    # block A's last row sits this far above contact 2's upper leg
        block_b_drop="120um",     # block B's first row sits this far below contact 2's lower leg
        # --- serpentines
        target_length="22mm",     # contact centre -> next contact centre
        serp_x_min="0.25mm",      # left turns of the serpentine rows
        serp_x_max="2.3mm",       # right-most x the serpentine rows may reach
        pitch="60um",             # row-to-row spacing (must be >= 2*fillet)
        fillet="30um",
        # --- chip / layout
        start_x="2.5mm",          # x of the chip's right edge (the left edge is its mirror)
        trace_width="10um",
        trace_gap="6um",
        layer="5",
        chip="main",
    )
    component_metadata = Dict(short_name="waveguide", _qgeometry_table_path="True")

    def get_points(self):
        return self._waveguide_points

    def make(self):
        p = self.p
        r, w, pitch, d, depth = p.fillet, p.trace_width, p.pitch, p.dip_width, p.contact_depth
        qx, qy = p.qubit_x, p.qubit_y
        pad_w, pad_h = p.qubit_pad_width, p.qubit_pad_height
        for name, val in (("pitch", pitch), ("dip_width", d),
                          ("corridor_drop", p.corridor_drop), ("block_b_drop", p.block_b_drop)):
            if val < 2 * r - 1e-12:
                raise ValueError(f"{name} must be >= 2*fillet")

        pad_half = qx + pad_w / 2
        # contact 1: right shoulder (dip from above)
        x_c1 = qx + p.contact_x
        x_a1, x_b1 = x_c1 - d / 2, x_c1 + d / 2                   # down-leg, up-leg
        y_dip1 = qy - depth
        # if x_a1 - w / 2 < qx + p.qubit_lead_width / 2 + p.trace_gap or x_b1 + w / 2 > pad_half:
        #     raise ValueError("Contact 1 legs must sit on the right shoulder: keep them between "
        #                      "the lead and the pad's right edge (adjust contact_x / dip_width).")
        # contact 2: right edge (dip from the right)
        yc2 = qy - p.side_contact_y
        yU, yL = yc2 + d / 2, yc2 - d / 2
        # if yU > y_dip1 - pitch + 1e-12:
        #     raise ValueError("Contact 2 sits too close to contact 1; increase side_contact_y.")
        # if yL - w / 2 < qy - pad_h:
        #     raise ValueError("Contact 2 runs off the bottom of the pad; reduce side_contact_y.")
        x_tip2 = pad_half - depth
        # corridor and contact 3: bottom edge (dip from below)
        x_cor = qx + p.corridor_x
        # if x_cor - w / 2 <= pad_half or x_cor + pitch > qx + p.serp_x_min + 1e-12:
        #     raise ValueError("corridor_x must lie between the pad's right edge and serp_x_min "
        #                      "(at least one pitch short of it).")
        x_p3, y_tip3 = qx + d / 2, qy - pad_h + depth
        # if d / 2 + w / 2 > pad_w / 2:
        #     raise ValueError("dip_width too wide for the bottom edge.")

        g = dict(qx=qx, start_x=p.start_x, pitch=pitch, x_c1=x_c1, x_a1=x_a1, x_b1=x_b1,
                 y_dip1=y_dip1, x_tip2=x_tip2, yc2=yc2, yU=yU, yL=yL, x_cor=x_cor,
                 x_min=qx + p.serp_x_min, x_p3=x_p3, y_tip3=y_tip3,
                 corridor_drop=p.corridor_drop, block_b_drop=p.block_b_drop)

        # --- size the two serpentine blocks so each link is exactly target_length.
        # Link length is linear in the row width W with slope n (n = rows), so for each
        # even n we probe at W = 1 mm and solve W in closed form; take the smallest n that
        # fits under serp_x_max.
        W_max = p.serp_x_max - p.serp_x_min

        def link_length(nA, WA, nB, WB, i):
            pts, cen = _layout(g, nA, WA, nB, WB)
            coords = _fillet_centerline(pts, r, ARC_POINTS)
            s = [_s_at_point(coords, c) for c in cen[i:i + 2]]
            return s[1] - s[0]

        def solve(i, nA_fixed=None, WA_fixed=None):
            for n in range(2, 400, 2):
                args = (n, 1.0, 2, 1.0) if i == 0 else (nA_fixed, WA_fixed, n, 1.0)
                W = 1.0 + (p.target_length - link_length(*args, i)) / n
                if W <= W_max:
                    break
            if W < 2 * r:
                raise ValueError("Serpentine rows would be shorter than 2*fillet; "
                                 "increase target_length or reduce serp_x_max.")
            return n, W

        nA, WA = solve(0)
        nB, WB = solve(1, nA, WA)

        points, centres = _layout(g, nA, WA, nB, WB)
        coords = _fillet_centerline(points, r, ARC_POINTS)
        _check_fillets(points, coords, r)
        cum = _arclength(coords)
        s_c = [_s_at_point(coords, c) for c in centres]

        # bookkeeping you can inspect from the notebook
        self._waveguide_points = np.array(points, dtype=float)
        self.coupling_centres = [tuple(c) for c in centres]
        self.coupling_spacings = list(np.diff(s_c))     # contact 1->2, 2->3, 3->4, 4->5
        self.rows_block_a, self.row_width_block_a = nA, WA
        self.rows_block_b, self.row_width_block_b = nB, WB
        self.length_total = cum[-1]

        centerline = draw.LineString(coords)
        self.add_qgeometry("path", {"waveguide": centerline},
                           width=w, fillet=0, layer=p.layer, chip=p.chip)
        self.add_qgeometry("path", {"waveguide_etch": centerline},
                           width=w + 2 * p.trace_gap, fillet=0,
                           layer=p.layer, subtract=True, chip=p.chip)

        # pins: outward normal = direction of travel at each end
        def pin(name, end, dirn):
            t = np.array([dirn[1], -dirn[0]])                # tangent so that normal == dirn
            self.add_pin(name, input_as_norm=False, width=w,
                         points=np.array([end - t * w / 2, end + t * w / 2]))

        pin("waveguide_connect", np.array(points[0], float), (1, 0))
        d_end = np.sign(points[-1][0] - points[-2][0])       # -1: leaves through the left edge
        pin("waveguide_end", np.array(points[-1], float), (d_end, 0))

In [35]:
component = myTransmon(design,"Q1",
    options={
        "trace_width": "25um",
        "taper_height": "00um",
        "lead_length": "1000um",
        "pad_width": "200um",
        "pad_height": "175um",
        "pad_gap": "50um",
        "orientation": "90",
        "pos_x": "0mm",
        "pos_y": "0mm",

    },
)


wg = GiantAtomWaveguide(design=design, name="waveguide")



gui.rebuild()
gui.autoscale()

print("coupling spacings (mm):", [f"{s:.4f}" for s in wg.coupling_spacings])
print(f"rows: block A {wg.rows_block_a}, block B {wg.rows_block_b}; total length {wg.length_total:.2f} mm")


coupling spacings (mm): ['22.0000', '22.0000', '22.0000', '22.0000']
rows: block A 12, block B 12; total length 94.42 mm


In [31]:
# ops = dict(fillet="60um")
# total_length = "22300um"
# options = Dict(total_length="22300um",
#             hfss_wire_bonds=True,
#             pin_inputs=Dict(
#                 start_pin=Dict(component="Q1", pin="top"), end_pin=Dict(component="Q1", pin="bottom")
#                 ),
#             width="10um",
#             gap="10um",
#             fillet="50um")
# try:
#     cpw.delete()
# except NameError:
#     pass

# cpw = RouteMeander(design, options=options)
# gui.rebuild()
# gui.autoscale()

